# Power BI report fix: the numbers

This notebook is the source of truth for every number in the README and in `powerbi/06-checks.md`.

1. Build the data both reports read (`data/sales_flat.csv`).
2. Compute, with plain SQL, every number each card, chart and table must show. The slow report and the fixed report must both match these.
3. Read the timings and model sizes measured in Power BI Desktop (`measurements/`) and work out how much faster and smaller the fixed report is.

Run it top to bottom from a fresh clone: `pip install -r requirements.txt`, then `jupyter nbconvert --to notebook --execute --inplace analysis/analysis.ipynb`.

In [1]:
%run ../prepare_data.py

Wrote C:\Users\DELL\GitHub\powerbi-report-fix\data\sales_flat.csv (2,098,633 rows, 801 MB)


In [2]:
import duckdb
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)
con = duckdb.connect()
con.sql("CREATE TABLE sales AS SELECT * FROM '../data/sales_flat.csv'")
q = lambda sql: con.sql(sql).df()

q('''
SELECT count(*)                        AS sales_rows,
       count(DISTINCT "Order Key")     AS orders,
       count(DISTINCT "Customer Key")  AS customers,
       count(DISTINCT "Product Key")   AS products,
       min("Order Date")               AS first_order,
       max("Order Date")               AS last_order,
       count(*) - count(DISTINCT ("Order Key", "Line Number")) AS duplicate_lines
FROM sales
''')

,sales_rows,orders,customers,products,first_order,last_order,duplicate_lines
0,2098633,875901,88063,2517,2015-01-01,2024-04-20,0


## What the report must show

The same measures as the report, written in SQL:

| Measure | Rule |
|---|---|
| Sales Amount | sum of Quantity × Net Price |
| Margin % | (Sales Amount − sum of Quantity × Unit Cost) ÷ Sales Amount |
| Orders, Customers | distinct order keys, distinct customer keys |
| Avg Order Value | Sales Amount ÷ Orders |
| Sales PY | Sales Amount for the same months one year earlier, with every other filter kept |
| Sales YoY % | (Sales Amount − Sales PY) ÷ Sales PY |
| Product Rank | rank of the product's Sales Amount among the products in the current filters, ties share a rank |

`kpis(where_now, where_py)` returns the six cards for any slicer selection.

In [3]:
def kpis(where_now, where_py):
    return q(f'''
    WITH now AS (
        SELECT sum("Quantity" * "Net Price")                     AS sales,
               sum("Quantity" * ("Net Price" - "Unit Cost"))     AS margin,
               count(DISTINCT "Order Key")                       AS orders,
               count(DISTINCT "Customer Key")                    AS customers
        FROM sales WHERE {where_now}
    ), py AS (
        SELECT sum("Quantity" * "Net Price") AS sales_py FROM sales WHERE {where_py}
    )
    SELECT round(sales, 2)                                   AS "Sales Amount",
           round(100 * (sales - sales_py) / sales_py, 1)     AS "Sales YoY %",
           round(100 * margin / sales, 1)                    AS "Margin %",
           orders                                            AS "Orders",
           round(sales / orders, 2)                          AS "Avg Order Value",
           customers                                         AS "Customers",
           round(sales_py, 2)                                AS "Sales PY"
    FROM now, py
    ''')

# Page 1, default selection: Year 2023, every country, every category.
kpis('year("Order Date") = 2023', 'year("Order Date") = 2022')

,Sales Amount,Sales YoY %,Margin %,Orders,Avg Order Value,Customers,Sales PY
0,"318,425,877.52",-28.40,56.00,159695,"1,993.96",62337,"444,516,718.72"


In [4]:
# Page 1, a filtered selection: Year 2022, Country Germany, Category Computers.
kpis('''year("Order Date") = 2022 AND "Country" = 'Germany' AND "Category" = 'Computers' ''',
     '''year("Order Date") = 2021 AND "Country" = 'Germany' AND "Category" = 'Computers' ''')

,Sales Amount,Sales YoY %,Margin %,Orders,Avg Order Value,Customers,Sales PY
0,"19,645,196.75",93.50,56.00,9854,"1,993.63",5197,"10,153,937.08"


In [5]:
# Page 1 chart: Sales Amount and Sales PY by month, Year 2023.
q('''
SELECT month("Order Date") AS month_number,
       strftime(min("Order Date"), '%b') AS month,
       round(sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2023)) AS sales_amount,
       round(sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2022)) AS sales_py
FROM sales
WHERE year("Order Date") IN (2022, 2023)
GROUP BY 1 ORDER BY 1
''')

,month_number,month,sales_amount,sales_py
0,1,Jan,"34,197,837.00","36,052,336.00"
1,2,Feb,"40,800,178.00","49,140,854.00"
2,3,Mar,"22,604,720.00","29,700,003.00"
3,4,Apr,"11,939,474.00","19,221,969.00"
4,5,May,"28,979,019.00","44,587,592.00"
5,6,Jun,"27,454,645.00","46,105,321.00"
6,7,Jul,"23,332,124.00","34,978,428.00"
7,8,Aug,"24,849,603.00","35,682,325.00"
8,9,Sep,"25,110,286.00","36,801,749.00"
9,10,Oct,"25,116,723.00","36,552,588.00"


In [6]:
# Page 1 bars: Sales Amount by Category and by Country, Year 2023.
by_category = q('''
SELECT "Category", round(sum("Quantity" * "Net Price")) AS sales_amount
FROM sales WHERE year("Order Date") = 2023 GROUP BY 1 ORDER BY 2 DESC
''')
by_country = q('''
SELECT "Country", round(sum("Quantity" * "Net Price")) AS sales_amount
FROM sales WHERE year("Order Date") = 2023 GROUP BY 1 ORDER BY 2 DESC
''')
display(by_category, by_country)

,Category,sales_amount
0,Computers,"113,112,204.00"
1,Cell phones,"59,754,615.00"
2,Home Appliances,"54,115,079.00"
3,TV and Video,"41,323,827.00"
4,"Music, Movies and Audio Books","21,768,661.00"
5,Cameras and camcorders,"18,765,677.00"
6,Audio,"6,847,569.00"
7,Games and Toys,"2,738,246.00"


,Country,sales_amount
0,United States,"156,132,838.00"
1,Germany,"41,279,015.00"
2,Canada,"37,241,921.00"
3,United Kingdom,"26,252,830.00"
4,Australia,"22,522,330.00"
5,Netherlands,"17,031,456.00"
6,France,"11,734,079.00"
7,Italy,"6,231,408.00"


In [7]:
# Page 2 table: the top 10 products by Sales Amount, Year 2023.
q('''
WITH p AS (
    SELECT "Product Name",
           sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2023)                       AS sales,
           sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2022)                       AS sales_py,
           sum("Quantity" * ("Net Price" - "Unit Cost")) FILTER (WHERE year("Order Date") = 2023)       AS margin,
           count(DISTINCT "Customer Key") FILTER (WHERE year("Order Date") = 2023)                      AS customers
    FROM sales WHERE year("Order Date") IN (2022, 2023)
    GROUP BY 1
)
SELECT "Product Name",
       round(sales)                                AS "Sales Amount",
       round(sales_py)                             AS "Sales PY",
       round(100 * (sales - sales_py) / sales_py, 1) AS "Sales YoY %",
       round(100 * margin / sales, 1)              AS "Margin %",
       customers                                   AS "Customers",
       rank() OVER (ORDER BY sales DESC)           AS "Product Rank"
FROM p WHERE sales IS NOT NULL
ORDER BY sales DESC LIMIT 10
''')

,Product Name,Sales Amount,Sales PY,Sales YoY %,Margin %,Customers,Product Rank
0,"Adventure Works 52"" LCD HDTV X590 White","2,317,817.00","2,712,426.00",-14.50,64.80,262,1
1,"Adventure Works 52"" LCD HDTV X590 Brown","2,264,428.00","2,867,452.00",-21.00,64.70,270,2
2,"Adventure Works 52"" LCD HDTV X590 Silver","2,038,838.00","3,218,644.00",-36.70,65.00,228,3
3,"Adventure Works 52"" LCD HDTV X590 Black","1,804,751.00","3,159,420.00",-42.90,64.30,207,4
4,Adventure Works Desktop PC2.33 XD233 White,"1,803,958.00","3,444,403.00",-47.60,64.80,621,5
5,Adventure Works Desktop PC2.33 XD233 Brown,"1,775,896.00","3,110,374.00",-42.90,64.60,623,6
6,WWI Desktop PC2.33 X2330 Silver,"1,733,712.00","3,152,473.00",-45.00,65.00,607,7
7,Adventure Works Desktop PC2.33 XD233 Silver,"1,712,446.00","3,158,969.00",-45.80,64.90,634,8
8,WWI Desktop PC2.33 X2330 Black,"1,678,976.00","3,077,120.00",-45.40,64.90,627,9
9,WWI Desktop PC2.33 X2330 Brown,"1,657,784.00","3,108,136.00",-46.70,64.80,615,10


In [8]:
# Page 2 matrix: Category rows with the grand total, Year 2023.
q('''
SELECT coalesce("Category", 'Total') AS "Category",
       round(sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2023)) AS "Sales Amount",
       round(100 * (sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2023)
                  - sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2022))
                  / sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2022), 1) AS "Sales YoY %",
       round(100 * sum("Quantity" * ("Net Price" - "Unit Cost")) FILTER (WHERE year("Order Date") = 2023)
                 / sum("Quantity" * "Net Price") FILTER (WHERE year("Order Date") = 2023), 1) AS "Margin %"
FROM sales WHERE year("Order Date") IN (2022, 2023)
GROUP BY ROLLUP ("Category")
ORDER BY "Category" = 'Total', 2 DESC
''')

,Category,Sales Amount,Sales YoY %,Margin %
0,Computers,"113,112,204.00",-36.10,56.20
1,Cell phones,"59,754,615.00",-27.70,54.00
2,Home Appliances,"54,115,079.00",-16.40,55.30
3,TV and Video,"41,323,827.00",-28.10,56.90
4,"Music, Movies and Audio Books","21,768,661.00",-22.30,58.60
5,Cameras and camcorders,"18,765,677.00",-21.10,58.20
6,Audio,"6,847,569.00",-10.40,55.10
7,Games and Toys,"2,738,246.00",-13.20,52.00
8,Total,"318,425,878.00",-28.40,56.00


## Before and after: timings and model size

Measured in Power BI Desktop by following `measurements/README.md`: one Performance Analyzer export per page per report, and one VertiPaq Analyzer export (`.vpax`, from DAX Studio) per report.

The cells that read them are added once the exports are in `measurements/`.